# Testing the implementation of biLSTM

## Load required libraries

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Masking, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from scikeras.wrappers import KerasClassifier
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
from tqdm import tqdm

# Preprocessing

In [2]:
# Load the normalized dataset
data = np.load("../preprocessing/ready/squat_sequences_all_features_normalized.npz")
X_data = data["X"]
y_labels = data["y"]


# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_labels, test_size=0.3, stratify=y_labels, random_state=42
)

print("✅ Dataset split summary:")
print(f"   X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}, y_test: {y_test.shape}")
print("   Label distribution in y:", np.bincount(y_labels))
print(y_test)


✅ Dataset split summary:
   X_train: (176, 189, 38), X_test: (76, 189, 38)
   y_train: (176,), y_test: (76,)
   Label distribution in y: [128 124]
[1 0 0 0 0 1 1 1 1 1 0 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 1 1 1 1 1 1 1
 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 1 1 1 0 1 1 0 0 1 1 1 1 0 0 0 0 1 1 1 1 1 0
 0 0]


# Defined model Function

In [3]:

def create_bilstm_model(input_shape, lstm_units=64, dropout_rate=0.3, learning_rate=0.001, **kwargs):
    """Creates and compiles a BiLSTM model with given hyperparameters."""
    
    model = Sequential([
        Input(shape=input_shape),
        Masking(mask_value=0.0), # Handles padding (ignores zeros) for the variable-length sequences
        Bidirectional(LSTM(lstm_units, return_sequences=True)), # Extracts temporal patterns from both directions
        Dropout(dropout_rate), # Regularization to prevent overfitting
        Bidirectional(LSTM(lstm_units // 2)),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')  # Outputs binary classification (good vs bad)
    ])
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# Random serach

In [4]:
import random

# Define the simple BiLSTM model
def create_bilstm_model(input_shape, lstm_units=128, dropout_rate=0.2, learning_rate=0.0005):
    model = Sequential([
        Input(shape=input_shape),
        Masking(mask_value=0.0),
        Bidirectional(LSTM(lstm_units)),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(learning_rate), loss="binary_crossentropy", metrics=["accuracy"])
    return model

# Define random search parameter space
param_space = {
    "lstm_units": lambda: random.randint(50, 150),
    "dropout_rate": lambda: random.uniform(0.2, 0.5),
    "learning_rate": lambda: random.uniform(0.0001, 0.001),
    "batch_size": lambda: random.randint(32, 64),
    "epochs": lambda: random.randint(50, 100)
}

# Run 20 random trials
results = []
for i in range(20):
    params = {k: sampler() for k, sampler in param_space.items()}
    print(f" Trial {i+1} with parameters: {params}")

    model = create_bilstm_model(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        lstm_units=params["lstm_units"],
        dropout_rate=params["dropout_rate"],
        learning_rate=params["learning_rate"]
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=params["epochs"],
        batch_size=params["batch_size"],
        verbose=0
    )

    best_val_accuracy = max(history.history["val_accuracy"])
    results.append({**params, "val_accuracy": best_val_accuracy})
    print(f" Best val accuracy: {best_val_accuracy:.4f}")

# Show sorted results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="val_accuracy", ascending=False)
print("\n🏆 Top 5 Configurations:")
print(results_df.head(5))

 Trial 1 with parameters: {'lstm_units': 141, 'dropout_rate': 0.46211629499799567, 'learning_rate': 0.0005550852226411262, 'batch_size': 50, 'epochs': 74}
 Best val accuracy: 0.9722
 Trial 2 with parameters: {'lstm_units': 101, 'dropout_rate': 0.36285453920170496, 'learning_rate': 0.0006064121529207488, 'batch_size': 63, 'epochs': 95}
 Best val accuracy: 0.9444
 Trial 3 with parameters: {'lstm_units': 72, 'dropout_rate': 0.4390427031798363, 'learning_rate': 0.00047490188492829683, 'batch_size': 41, 'epochs': 86}
 Best val accuracy: 0.9722
 Trial 4 with parameters: {'lstm_units': 98, 'dropout_rate': 0.2850504567226487, 'learning_rate': 0.0007922202854821405, 'batch_size': 61, 'epochs': 73}
 Best val accuracy: 0.9722
 Trial 5 with parameters: {'lstm_units': 60, 'dropout_rate': 0.35766344373409464, 'learning_rate': 0.00034673089051903716, 'batch_size': 53, 'epochs': 88}
 Best val accuracy: 0.9722
 Trial 6 with parameters: {'lstm_units': 100, 'dropout_rate': 0.23090100489163998, 'learning_

# Train the final model with the best hyperparameters

In [5]:
# Apparently this only outputs the best parameters from the last fold and not the best parameters overall (most common over all folds) 
# print(f"\n Best overall parameters: {grid_search.best_params_}")

best_config = {
    "lstm_units": 52,
    "dropout_rate": 0.448,
    "learning_rate": 0.000533,
    "batch_size": 43,
    "epochs": 61
}


final_model = create_bilstm_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    lstm_units=best_config["lstm_units"],
    dropout_rate=best_config["dropout_rate"],
    learning_rate=best_config["learning_rate"]
)

final_model.fit(
    X_train, y_train,
    epochs=best_config["epochs"],
    batch_size=best_config["batch_size"],
    verbose=1
)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)
print(f"✅ Final test accuracy: {accuracy:.4f}")

Epoch 1/61
5/5 [==============================] - 10s 45ms/step - loss: 0.6895 - accuracy: 0.5795
Epoch 2/61
5/5 [==============================] - 0s 27ms/step - loss: 0.6809 - accuracy: 0.5455
Epoch 3/61
5/5 [==============================] - 0s 28ms/step - loss: 0.7063 - accuracy: 0.5057
Epoch 4/61
5/5 [==============================] - 0s 24ms/step - loss: 0.6547 - accuracy: 0.6080
Epoch 5/61
5/5 [==============================] - 0s 27ms/step - loss: 0.6481 - accuracy: 0.6193
Epoch 6/61
5/5 [==============================] - 0s 25ms/step - loss: 0.6284 - accuracy: 0.6648
Epoch 7/61
5/5 [==============================] - 0s 26ms/step - loss: 0.6213 - accuracy: 0.6761
Epoch 8/61
5/5 [==============================] - 0s 25ms/step - loss: 0.6058 - accuracy: 0.6875
Epoch 9/61
5/5 [==============================] - 0s 28ms/step - loss: 0.5836 - accuracy: 0.7102
Epoch 10/61
5/5 [==============================] - 0s 27ms/step - loss: 0.5605 - accuracy: 0.7386
Epoch 11/61
5/5 [===========

# Evaluation

In [6]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Get predictions and convert probabilities to binary labels
y_pred_probs = final_model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int")

# Print classification report
print(classification_report(y_test, y_pred))

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Compute ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_probs)
print(f" ROC-AUC Score: {roc_auc:.4f}")

3/3 [==============================] - 4s 27ms/step
              precision    recall  f1-score   support

           0       0.97      0.90      0.93        39
           1       0.90      0.97      0.94        37

    accuracy                           0.93        76
   macro avg       0.94      0.94      0.93        76
weighted avg       0.94      0.93      0.93        76

Confusion Matrix:
 [[35  4]
 [ 1 36]]
 ROC-AUC Score: 0.9861
